In [1]:
import re, unicodedata, difflib, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd
from scipy.optimize import linear_sum_assignment

RAW, INT, CLEAN = Path("../raw"), Path("../interim"), Path("../clean")
TODAY = dt.date.today().isoformat()
SCOPE_STATES = ["Bihar", "Maharashtra", "Tamil Nadu", "Uttar Pradesh"]
YEARS = [2014, 2019, 2024]

raw = pd.read_csv(RAW / "lok_sabha_candidate_dfs" / "candidates_all_years.csv")            # reference only, never edited
cand = pd.read_csv(INT / "candidates_processed.csv")           # MyNeta, working copy
oc = pd.read_csv(CLEAN / "opencity_results_clean_2026-10-20.csv")
tcpd = pd.read_csv(INT / "TCPD-PPI_1962_2021.csv")

LOG = []
def log(step, before, after, note=""):
    LOG.append(dict(step=step, count_before=before, count_after=after, rows_affected=abs(before - after), note=note))
    print(f"{step}: {before} -> {after} {note}")

print(cand.shape, oc.shape, tcpd.shape)
key = ["year", "state", "const", "sno", "candidate"]
assert (cand[key].values == raw[key].values).all()             # processed and raw are row-aligned

(10395, 15) (10378, 23) (10153, 21)


In [2]:
cand["state"] = cand["state"].str.title()
assert set(cand.state) == set(SCOPE_STATES) and set(cand.year) == set(YEARS)

bye = cand[cand.is_byelection].copy()
bye.to_csv(INT / f"myneta_byelections_{TODAY}.csv", index=False)
n0 = len(cand); cand = cand[~cand.is_byelection].copy()
log("drop by-election rows (OpenCity already excludes them)", n0, len(cand), "saved to interim/myneta_byelections_*.csv")

def rupees(s):
    s = str(s).strip()
    if s.lower() == "nil": return 0
    m = re.search(r"Rs\s*([\d,]+)", s)
    return int(m.group(1).replace(",", "")) if m else pd.NA

for c in ["total_assets", "liabilities"]:
    before = cand[c].astype(str).str.contains(r"[\r\n~]").sum()
    cand[c + "_rs"] = cand[c].map(rupees).astype("Int64")
    log(f"{c}: strip \\r\\n/~ text, parse to integer rupees", before, int(cand[c + "_rs"].isna().sum()), "cells with junk -> unparsed left")
cand = cand.drop(columns=["total_assets", "liabilities"]).rename(columns={"total_assets_rs": "total_assets", "liabilities_rs": "liabilities"})

n_age0 = int((cand.age == 0).sum())
cand["age"] = cand["age"].replace(0, pd.NA).astype("Int64")
log("age 0 -> missing (not a real age)", n_age0, int((cand.age == 0).sum()))
cand["criminal_cases"] = cand["criminal_cases"].astype("Int64")
edu = ["Illiterate", "Literate", "5th Pass", "8th Pass", "10th Pass", "12th Pass", "Graduate", "Graduate Professional", "Post Graduate", "Doctorate"]
cand["education"] = cand["education"].replace({"Not Given": pd.NA, "Others": pd.NA})
cand["education"] = pd.Categorical(cand["education"], categories=edu, ordered=True)
cand["election_date"] = pd.to_datetime(cand["election_date"], format="%d-%m-%Y")
cand = cand.drop(columns=["is_byelection", "bye_election_date"])
print(cand.dtypes)

drop by-election rows (OpenCity already excludes them): 10395 -> 10228 saved to interim/myneta_byelections_*.csv
total_assets: strip \r\n/~ text, parse to integer rupees: 10142 -> 0 cells with junk -> unparsed left
liabilities: strip \r\n/~ text, parse to integer rupees: 8437 -> 0 cells with junk -> unparsed left
age 0 -> missing (not a real age): 39 -> 0 
year                       int64
state                        str
const                        str
sno                        int64
candidate                    str
is_winner                   bool
party                        str
criminal_cases             Int64
education               category
age                        Int64
election_date     datetime64[us]
total_assets               Int64
liabilities                Int64
dtype: object


In [3]:
def ckey(s):
    s = re.sub(r"\s*\((SC|ST)\)\s*$", "", str(s).strip().upper())
    return re.sub(r"[^A-Z]", "", s)                       # drops spaces, hyphens, dots

cand["const_raw"] = cand["const"]
cand["myneta_suffix"] = cand["const"].str.extract(r"\((SC|ST)\)\s*$")[0]   # not reliable, see assert below
cand["ck"] = cand["const"].map(ckey)
oc["ck"] = oc["Constituency_Name"].map(ckey)

oc_seats = oc[["Year", "State", "ck", "Constituency_No", "Constituency_Name", "Constituency_Type"]].drop_duplicates()
cw = {}
for (y, s), g in cand.groupby(["year", "state"]):
    tk = set(oc_seats[(oc_seats.Year == y) & (oc_seats.State == s)].ck)
    for k in set(g.ck) - tk:
        hit = difflib.get_close_matches(k, sorted(tk - set(g.ck)), n=1, cutoff=0.85)
        if hit: cw[(y, s, k)] = hit[0]
print(pd.DataFrame([(*k, v) for k, v in cw.items()], columns=["year", "state", "myneta_key", "opencity_key"]).to_string())  # eyeball: same place?
cand["ck"] = [cw.get((y, s, k), k) for y, s, k in zip(cand.year, cand.state, cand.ck)]
log("constituency spellings mapped to OpenCity's", len(cw), len(cw), "seat-year pairs (table above)")

cand = cand.merge(oc_seats.rename(columns={"Year": "year", "State": "state", "Constituency_Type": "oc_type"}), on=["year", "state", "ck"], how="left")
assert cand.Constituency_No.notna().all(), "unmapped constituencies"
assert (cand.myneta_suffix.isna() | (cand.myneta_suffix == cand.oc_type)).all(), "SC/ST suffix disagrees with OpenCity"
print("rows with no suffix in MyNeta but SC/ST in OpenCity:", int((cand.myneta_suffix.isna() & (cand.oc_type != "GEN")).sum()))
cand = cand.rename(columns={"oc_type": "Constituency_Type"}).drop(columns=["const", "myneta_suffix"])
print(cand.groupby(["year", "state"]).Constituency_No.nunique().unstack())     # expect 40/48/39/80 each year

   year          state         myneta_key       opencity_key
0  2014    Maharashtra         AHMEDNAGAR         AHMADNAGAR
1  2014     Tamil Nadu         TIRUVALLUR        THIRUVALLUR
2  2014  Uttar Pradesh  GAUTAMBUDHHANAGAR  GAUTAMBUDDHANAGAR
3  2019    Maharashtra         AHMEDNAGAR         AHMADNAGAR
4  2019     Tamil Nadu         TIRUVALLUR        THIRUVALLUR
5  2024    Maharashtra       HATKANANGALE        HATKANANGLE
6  2024    Maharashtra         AHMEDNAGAR         AHMADNAGAR
7  2024     Tamil Nadu         TIRUVALLUR        THIRUVALLUR
constituency spellings mapped to OpenCity's: 8 -> 8 seat-year pairs (table above)
rows with no suffix in MyNeta but SC/ST in OpenCity: 987
state  Bihar  Maharashtra  Tamil Nadu  Uttar Pradesh
year                                                
2014      40           48          39             80
2019      40           48          39             80
2024      40           48          39             80


In [4]:
tl = tcpd[(tcpd.Assembly == "Lok_Sabha") & (tcpd.State_Name == "All_States")].copy()
tl["Party_ID"] = tl["Party_ID"].astype(int)
assert tl.Party_ID.is_unique
print("TCPD last year covered:", int(tl.Last_Year.max()), "-> 2024-only parties cannot have a TCPD ID")

def pk(s):
    s = unicodedata.normalize("NFKD", str(s)).lower().replace("&", "and").replace("–", "-")
    return re.sub(r"[^a-z0-9]", "", s)

lookup, clash = {}, {}
for _, r in tl.sort_values(["Last_Year", "Candidates_Contested"]).iterrows():     # later/larger party wins a clash
    for v in {r.Party_Name, r.Frequent_Abbreviation, r.Last_Abbreviation, *str(r.Abbreviations).split("|")}:
        if pk(v) in lookup and lookup[pk(v)] != r.Party_ID: clash.setdefault(pk(v), {lookup[pk(v)]}).add(r.Party_ID)
        lookup[pk(v)] = r.Party_ID
print("keys that point to more than one TCPD party:", len(clash))

MANUAL = {"congress": "INC", "indiannationalcongress": "INC", "aiadmk": "ADMK", "aap": "AAAP", "cpim": "CPM", "cpi(m)": "CPM",
          "shivsena": "SHS", "nationalistcongressparty": "NCP", "communistpartyofindia": "CPI", "jdu": "JD(U)",
          "allindiamajliseittehadulmuslimeen": "AIMIM"}
for a, b in MANUAL.items():
    if pk(b) in lookup: lookup[pk(a)] = lookup[pk(b)]
tname = tl.set_index("Party_ID")

def to_id(*labels):
    for lab in labels:
        if pk(lab) in lookup: return lookup[pk(lab)], "TCPD"
    for lab in labels:                                        # typo-level differences only
        h = difflib.get_close_matches(pk(lab), list(lookup), n=1, cutoff=0.9)
        if h: return lookup[h[0]], "TCPD_fuzzy"
    return pd.NA, "none"

labels = pd.concat([cand[["party"]].rename(columns={"party": "a"}).assign(b=pd.NA),
                    oc[["Party_Full", "Party"]].rename(columns={"Party_Full": "a", "Party": "b"})]).drop_duplicates()
cw_p = {(a, b): to_id(*[x for x in (a, b) if pd.notna(x) and x != "OTHER"]) for a, b in labels.itertuples(index=False)}

unmatched = sorted({pk(a) for (a, b), (i, src) in cw_p.items() if src == "none"})
synthetic = {k: 900000 + n for n, k in enumerate(unmatched)}   # parties TCPD does not have
def party_id(a, b=pd.NA):
    i, src = cw_p.get((a, b), to_id(a, b))
    return (i, src) if src != "none" else (synthetic.get(pk(a), pd.NA), "synthetic")

cand[["Party_ID", "Party_ID_Source"]] = [party_id(a) for a in cand.party]
oc[["Party_ID", "Party_ID_Source"]] = [party_id(a, b) for a, b in zip(oc.Party_Full, oc.Party)]
print(cand.groupby(["year", "Party_ID_Source"]).size().unstack(fill_value=0))
print(oc.groupby(["Year", "Party_ID_Source"]).size().unstack(fill_value=0))
print(oc[oc.Party_ID_Source == "TCPD_fuzzy"][["Party_Full", "Party"]].drop_duplicates())    # eyeball fuzzy matches

TCPD last year covered: 2019 -> 2024-only parties cannot have a TCPD ID
keys that point to more than one TCPD party: 105
Party_ID_Source  TCPD  TCPD_fuzzy  synthetic
year                                        
2014             3515          76         20
2019             3156          58         18
2024             2867          28        490
Party_ID_Source  TCPD  TCPD_fuzzy  synthetic
Year                                        
2014             3637           0          0
2019             3322           0          0
2024             2874          26        519
                               Party_Full  Party
7291              Rashtriya Sanatan Party  OTHER
7351               Bhartiya Manavta Party  OTHER
7392                 Rashtriya Lok Morcha  OTHER
7407              Rashtriya Jan Jan Party  OTHER
7514   Republican Party of India (Social)  OTHER
7617            Republican Party of India  OTHER
7842                Hindustan Janta Party  OTHER
7916                Bhartiya Asmita P

In [5]:
TITLES = r"\b(DR|ADV|ADVOCATE|MD|MOHD|MOHAMMAD|MOHAMMED|MO|SHRI|SRI|SMT|SHRIMATI|KUMARI|KU|PROF|PANDIT|PT|CA|CAPT|COL|LT|MAJOR|RTD|ER|ENGR|MRS|MR|MS|HAJI|ALHAJ|ALIAS)\b"
def name_tokens(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().upper()
    s = re.sub(r"[^A-Z ]", " ", s)
    return re.sub(TITLES, " ", s).split()

cand["Candidate_Raw"] = cand["candidate"]
cand["Candidate"] = cand["candidate"].map(lambda s: " ".join(re.sub(r"[.,]", " ", str(s)).upper().split()))
cand["name_key"] = cand["candidate"].map(lambda s: " ".join(sorted(name_tokens(s))))
oc["name_key"] = oc["Candidate"].map(lambda s: " ".join(sorted(name_tokens(s))))
cand = cand.drop(columns=["candidate"])

seat = ["year", "state", "Constituency_No"]
print("same name_key in one seat, MyNeta:", int(cand.duplicated(seat + ["name_key"]).sum()),
      "| OpenCity:", int(oc.rename(columns={"Year": "year", "State": "state"}).duplicated(seat + ["name_key"]).sum()))

same name_key in one seat, MyNeta: 20 | OpenCity: 32


In [6]:
g = cand.groupby(seat).sno.agg(n="size", mx="max", nu="nunique")
gaps = g[(g.mx > g.nu) | (g.n != g.nu)]
print("seats with sno holes or duplicates:\n", gaps)

def sim(a, b):
    ta, tb = a.split(), b.split()
    whole = difflib.SequenceMatcher(None, a, b).ratio()
    part = lambda X, Y: np.mean([max([difflib.SequenceMatcher(None, x, y).ratio() for y in Y] or [0]) for x in X]) if X else 0
    return max(whole, (part(ta, tb) + part(tb, ta)) / 2)

cand = cand.reset_index(drop=True); oc = oc.reset_index(drop=True)
pairs = []
oc_seat = oc.rename(columns={"Year": "year", "State": "state"})
for (y, s, c), A in cand.groupby(seat):
    B = oc_seat[(oc_seat.year == y) & (oc_seat.state == s) & (oc_seat.Constituency_No == c)]
    M = np.zeros((len(A), len(B)))
    for i, a in enumerate(A.itertuples()):
        for j, b in enumerate(B.itertuples()):
            sc = sim(a.name_key, b.name_key) + (0.5 if a.name_key == b.name_key else 0)
            if pd.notna(a.Party_ID) and pd.notna(b.Party_ID) and a.Party_ID_Source != "synthetic" and b.Party_ID_Source != "synthetic":
                sc += 0.3 if a.Party_ID == b.Party_ID else -0.1
            sc += 0.2 if (a.is_winner and b.Winner) else 0
            M[i, j] = sc
    for i, j in zip(*linear_sum_assignment(-M)):        # one-to-one best assignment inside the seat
        pairs.append((A.index[i], B.index[j], M[i, j]))
pairs = pd.DataFrame(pairs, columns=["ci", "oi", "score"])
print(pd.cut(pairs.score, [-1, .6, .7, .8, 1.0, 3]).value_counts().sort_index())

rev = pairs.merge(cand[["year", "Candidate", "party"]], left_on="ci", right_index=True).merge(oc[["Candidate", "Party_Full"]], left_on="oi", right_index=True, suffixes=("_myneta", "_oc"))
print(rev[rev.score < 0.8].sort_values("score")[["year", "Candidate_myneta", "Candidate_oc", "party", "Party_Full", "score"]].to_string())

# weak name match AND different real party -> probably two different people: break the pair
real = lambda d, i: d.loc[i, "Party_ID_Source"].isin(["TCPD", "TCPD_fuzzy"]).values
differ = real(cand, pairs.ci) & real(oc, pairs.oi) & (cand.loc[pairs.ci, "Party_ID"].values != oc.loc[pairs.oi, "Party_ID"].values)
broken = pairs[(pairs.score < 0.8) & differ]
print("pairs broken:", len(broken))
pairs = pairs.drop(broken.index)
orphans = cand.drop(pairs.ci)
orphans.drop(columns=["ck", "name_key"]).to_csv(INT / f"myneta_unmatched_{TODAY}.csv", index=False)
log("MyNeta rows matched to an OpenCity row", len(cand), len(pairs), "unmatched saved to interim/myneta_unmatched_*.csv")
assert pairs.score.min() > 0.5
assert (cand.loc[pairs.ci, "is_winner"].values == oc.loc[pairs.oi, "Winner"].values).all(), "winner flag disagrees"

seats with sno holes or duplicates:
                                    n  mx  nu
year state       Constituency_No            
2019 Maharashtra 2                 9  26   9
2024 Maharashtra 34               14  19  14
score
(-1.0, 0.6]        2
(0.6, 0.7]         2
(0.7, 0.8]         6
(0.8, 1.0]        83
(1.0, 3.0]     10135
Name: count, dtype: int64
      year                       Candidate_myneta                           Candidate_oc                                party          Party_Full     score
6028  2019                                 RAMPAL                                RAM PAL               Rashtriya Samaj Paksha                RSPS  0.566667
6695  2019                      RAJESH KUMAR GOND                                 RAJESH             Gondvana Gantantra Party                 IND  0.596970
4527  2019                       MOHAMMAD MOHASIN                  MOHSIN SIR NASIM BHAI               Navbharat Nirman Party                 NNP  0.657605
6279  2019            

In [7]:
m = pairs.set_index("oi")
mn_cols = ["Candidate_Raw", "sno", "criminal_cases", "education", "age", "total_assets", "liabilities", "election_date", "party", "Party_ID", "Party_ID_Source"]
mn = cand.loc[m.ci.values, mn_cols].copy(); mn.index = m.index
mn = mn.rename(columns={"party": "Party_MyNeta", "Party_ID": "Party_ID_MyNeta", "Party_ID_Source": "Party_ID_Source_MyNeta"})
merged = oc.join(mn)
merged["In_MyNeta"] = merged.index.isin(m.index)
merged["Match_Score"] = m.score.reindex(merged.index).round(3)

conflict = merged[(merged.Party_ID_Source != "synthetic") & (merged.Party_ID_Source_MyNeta.isin(["TCPD", "TCPD_fuzzy"])) & (merged.Party_ID != merged.Party_ID_MyNeta)]
print("party ID disagreements OpenCity vs MyNeta:", len(conflict))
print(conflict[["Year", "Candidate", "Party_Full", "Party_MyNeta"]].drop_duplicates(["Party_Full", "Party_MyNeta"]).to_string())
merged["Party_Conflict"] = merged.index.isin(conflict.index)     # flagged for review, not auto-fixed
use_mn = (merged.Party_ID_Source == "synthetic") & merged.Party_ID_Source_MyNeta.isin(["TCPD", "TCPD_fuzzy"])
merged.loc[use_mn, "Party_ID"] = merged.loc[use_mn, "Party_ID_MyNeta"]
merged.loc[use_mn, "Party_ID_Source"] = merged.loc[use_mn, "Party_ID_Source_MyNeta"]
merged["Party_ID"] = merged["Party_ID"].astype("Int64")
merged["Party_TCPD_Name"] = merged.Party_ID.map(tname["Party_Name"].str.strip().str.title())
merged["Party_TCPD_Abbr"] = merged.Party_ID.map(tname["Frequent_Abbreviation"])

missing = merged[~merged.In_MyNeta]
print("OpenCity candidates with no MyNeta row:", len(missing))
print(missing.groupby(["Year", "State"]).size().unstack(fill_value=0))
print("explained by an sno hole:", int(sum(gaps.mx - gaps.nu)), "of", len(missing))

log("OpenCity candidates absent from MyNeta (In_MyNeta=False)", len(oc), int(merged.In_MyNeta.sum()), "kept: votes exist, curation columns are NA")
log("party ID from TCPD (not synthetic)", len(merged), int(merged.Party_ID_Source.isin(["TCPD", "TCPD_fuzzy"]).sum()), "rest are post-2019 parties")
merged["MyNeta_sno"] = merged["sno"].astype("Int64")
merged = merged.drop(columns=["sno", "ck", "name_key", "Party_ID_MyNeta", "Party_ID_Source_MyNeta"])
merged = merged.rename(columns={"criminal_cases": "Criminal_Cases", "education": "Education", "age": "Age",
                                "total_assets": "Total_Assets_Rs", "liabilities": "Liabilities_Rs", "election_date": "Election_Date"})

party ID disagreements OpenCity vs MyNeta: 46
      Year                      Candidate   Party_Full                         Party_MyNeta
849   2014    GHAYWAN ROSHAN JAMBUWANTRAO        RP(K)          Republican Paksha (Khoripa)
1258  2014       MAHADEV JAGANNATH JANKAR         RSPS               Rashtriya Samaj Paksha
1490  2014           MHAMANE ANIL SUBHASH          MVA                                  IND
1496  2014          YUVRAJ SHIVAJI KAMBLE         BMUP                                  IND
1544  2014                      PAULRAJ T         MMKA                 Makkal Manadu Katchi
2502  2014                         YUNUSH          NNP               Navbharat Nirman Party
2610  2014                    CHANDRA PAL        A S P                   Adarsh Samaj Party
2950  2014          KUWAR HARIVANSH SINGH           AD                             Apna Dal
3564  2014               RAM LAKHAN GUPTA          JAP                      Jantantra Party
3721  2019                  SHAKEE

In [8]:
keys = ["Year", "State", "Constituency_No", "Person_ID"]
assert not merged.duplicated(keys).any(), "duplicate key"
assert merged.groupby(["Year", "State", "Constituency_No"]).Winner.sum().eq(1).all(), "a seat without exactly one winner"
assert len(merged) == len(oc)
print("assets < liabilities (check only):", int((merged.Total_Assets_Rs < merged.Liabilities_Rs).sum()))
print(merged.loc[merged.Criminal_Cases > 50, ["Year", "Candidate", "Criminal_Cases"]].to_string())

la = np.log10(merged.Total_Assets_Rs.astype("float").where(merged.Total_Assets_Rs > 0))   # flag, don't delete
q1 = la.groupby(merged.Year).transform(lambda s: s.quantile(.25)); q3 = la.groupby(merged.Year).transform(lambda s: s.quantile(.75))
merged["Assets_Outlier"] = np.where(la > q3 + 3 * (q3 - q1), "high", np.where(la < q1 - 3 * (q3 - q1), "low", "none"))
print(merged.Assets_Outlier.value_counts())

out_int = INT / f"myneta_candidates_clean_{TODAY}.csv"
out_clean = CLEAN / f"loksabha_candidates_merged_{TODAY}.csv"
cand.drop(columns=["ck", "name_key"]).to_csv(out_int, index=False)
merged.to_csv(out_clean, index=False)
pd.DataFrame(LOG).to_csv(CLEAN / f"merge_cleaning_log_{TODAY}.csv", index=False)

back = pd.read_csv(out_clean, parse_dates=["Election_Date"], dtype={"Total_Assets_Rs": "Int64", "Liabilities_Rs": "Int64", "Age": "Int64", "Criminal_Cases": "Int64", "Party_ID": "Int64"})
back["Education"] = pd.Categorical(back["Education"], categories=edu, ordered=True)     # CSV forgets categories
assert (back.Total_Assets_Rs.fillna(-1) == merged.Total_Assets_Rs.fillna(-1)).all()
print(back.dtypes)

assets < liabilities (check only): 231
      Year       Candidate  Criminal_Cases
2269  2014   PUSHPARAYAN M             380
2329  2014  UDAYAKUMAR S P             382
6916  2019     ATEEK AHMAD              59
Assets_Outlier
none    10377
low         1
Name: count, dtype: int64


State                               str
Constituency_No                   int64
Constituency_Name                   str
Constituency_Type                   str
Year                              int64
Person_ID                         int64
Candidate                           str
Candidate_Key                       str
Namesake_In_Seat                   bool
Party                               str
Party_Full                          str
Party_Group                         str
Sex                                 str
Incumbent                          bool
Incumbent_Source                    str
Position                          int64
Winner                             bool
Votes                             int64
Vote_Share_Percentage           float64
Margin                            int64
Seat_Valid_Votes                  int64
Seat_Candidates                   int64
Votes_Outlier                       str
Party_ID                          Int64
Party_ID_Source                     str


In [9]:
fz = pd.concat([oc[oc.Party_ID_Source == "TCPD_fuzzy"][["Party_Full", "Party_ID"]].rename(columns={"Party_Full": "label"}),
                cand[cand.Party_ID_Source == "TCPD_fuzzy"][["party", "Party_ID"]].rename(columns={"party": "label"})]).drop_duplicates()
fz["TCPD_name"] = fz.Party_ID.map(tname.Party_Name)
print(fz.to_string())

                                               label Party_ID                                                TCPD_name
7291                         Rashtriya Sanatan Party     6594                                    Rashtriya Janta Party
7351                          Bhartiya Manavta Party     1605                                   Bharatiya Janata Party
7392                            Rashtriya Lok Morcha    12187                                Rashtriya Lok Seva Morcha
7407                         Rashtriya Jan Jan Party     6594                                    Rashtriya Janta Party
7514              Republican Party of India (Social)    12285                            Republican Party of India (A)
7617                       Republican Party of India    12285                            Republican Party of India (A)
7842                           Hindustan Janta Party    10341                                   HINDUSTAN JANATA PARTY
7916                           Bhartiya Asmita P

In [10]:
print(missing.groupby(["Year", "State", "Constituency_Name"]).size().sort_values(ascending=False).head(15))

Year  State          Constituency_Name
2024  Maharashtra    PUNE                 21
2019  Maharashtra    DHULE                19
      Tamil Nadu     ARAKKONAM             4
                     TIRUVANNAMALAI        4
2024  Maharashtra    NASHIK                4
2019  Tamil Nadu     VELLORE               4
                     KALLAKURICHI          3
                     DINDIGUL              3
      Bihar          PATALIPUTRA           2
      Tamil Nadu     CHIDAMBARAM           2
      Uttar Pradesh  GHOSI                 2
                     HAMIRPUR              2
                     AKBARPUR              2
      Maharashtra    PUNE                  2
      Tamil Nadu     KRISHNAGIRI           2
dtype: int64


In [11]:
MANUAL["dravidamunnetrakazhagam"] = "DMK"          # TCPD spells it "MUMMETRA"; the 0.94 cutoff would drop it
for a, b in MANUAL.items():
    if pk(b) in lookup: lookup[pk(a)] = lookup[pk(b)]
tname = tl.set_index("Party_ID")

REJECT = {"republicanpartyofindia"}                 # TCPD only has RPI(A); the plain name may be another RPI

def fuzzy(k):
    if k in REJECT: return None
    for h in difflib.get_close_matches(k, list(lookup), n=3, cutoff=0.9):
        r = difflib.SequenceMatcher(None, k, h).ratio()
        trunc = (h.startswith(k) or k.startswith(h)) and min(len(k), len(h)) >= 25   # MyNeta cuts labels at 45 chars
        if r >= 0.94 or trunc: return h
    return None

def to_id(*labels):
    for lab in labels:
        if pk(lab) in lookup: return lookup[pk(lab)], "TCPD"
    for lab in labels:
        h = fuzzy(pk(lab))
        if h: return lookup[h], "TCPD_fuzzy"
    return pd.NA, "none"

In [13]:
chk = raw[((raw.year == 2024) & raw.const.str.startswith("PUNE")) | ((raw.year == 2019) & raw.const.str.startswith("DHULE"))]
for (y, c, path), g in chk.groupby(["year", "const", "html_path"]):
    html = Path(path.replace("\\", "/")).read_text(encoding="utf-8", errors="ignore")
    print(y, c, "ids in html:", len(set(re.findall(r"candidate_id=(\d+)", html))), "| rows kept:", len(g))

2019 DHULE ids in html: 9 | rows kept: 9
2024 PUNE ids in html: 14 | rows kept: 14


In [14]:
from scipy import stats
miss, have = merged[~merged.In_MyNeta], merged[merged.In_MyNeta]
print(merged.groupby("In_MyNeta").agg(n=("Votes", "size"), median_votes=("Votes", "median"),
      median_share=("Vote_Share_Percentage", "median"), winners=("Winner", "sum")))
print(pd.crosstab(merged.Party_Group, merged.In_MyNeta, normalize="columns").round(3))
print("Mann-Whitney vote share p =", stats.mannwhitneyu(miss.Vote_Share_Percentage, have.Vote_Share_Percentage).pvalue)
print("chi2 party group p =", stats.chi2_contingency(pd.crosstab(merged.Party_Group, merged.In_MyNeta))[1])
print("chi2 year p =", stats.chi2_contingency(pd.crosstab(merged.Year, merged.In_MyNeta))[1])
print("missing rate by year:\n", merged.groupby("Year").In_MyNeta.apply(lambda s: (~s).mean()).round(4))

               n  median_votes  median_share  winners
In_MyNeta                                            
False        156        1882.0          0.17        3
True       10222        2953.5          0.28      618
In_MyNeta    False  True 
Party_Group              
Independent  0.513  0.450
Main party   0.103  0.193
Other party  0.385  0.357
Mann-Whitney vote share p = 1.9758505848709395e-06
chi2 party group p = 0.015904544975522596
chi2 year p = 3.686312739758579e-14
missing rate by year:
 Year
2014    0.0071
2019    0.0286
2024    0.0102
Name: In_MyNeta, dtype: float64


In [15]:
print(merged.loc[~merged.In_MyNeta & merged.Winner, ["Year", "State", "Constituency_Name", "Candidate", "Party_Full"]])

      Year          State  Constituency_Name                     Candidate  \
1104  2014    Maharashtra  MUMBAI NORTH WEST  GAJANAN CHANDRAKANT KIRTIKAR   
2965  2014  Uttar Pradesh        FARRUKHABAD                 MUKESH RAJPUT   
6501  2019  Uttar Pradesh           AKBARPUR          DEVENDRA SINGH BHOLE   

     Party_Full  
1104        SHS  
2965        BJP  
6501        BJP  
